In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import setlistfm
mio   = setlistfm.MusicDBIO(verbose=False, mkDirs=False)
apiio = setlistfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant SetListFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/SetListFM


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

SetListFM Search Results
   Global Master Search:      23504
   Global Master Search Data: 1400
   Search Artists:            22104
   Errors:                    85
   Known Summary IDs:         0


# Search For New Artists

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=False,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [9]:
from musicdb import PanDBIO
from gate import IOStore

pdbio = PanDBIO()
mmeDF = pdbio.getData().sort_values(by=["Counts", "Albums"], ascending=False)

ios     = IOStore()
mdbio   = ios.get(db="MusicBrainz")
refData = mdbio.data.getSummaryRefData().to_dict()

mbIDData = mmeDF[mmeDF["MusicBrainz"].notna()][["ArtistName", "MusicBrainz"]]
mbIDData["MBRef"] = mbIDData["MusicBrainz"].apply(refData.get).apply(lambda x: x.split('/')[-1] if isinstance(x,str) else None)

searchedForMasterArtists = masterArtists.get()
artistNamesToGet = mbIDData[~mbIDData["MusicBrainz"].isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(mbIDData.shape[0]))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

del pdbio
del mmeDF
del refData
del mbIDData

#   Artist Names To Get:          793373

SetListFM Search Results
   Known Master Artist Names:    814451
   Known Spotify Artist Names:   23504
   Artist Names To Get:          790884


## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

## Run @ 3-4 PM every day

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:50pm")
n    = 0
maxN = 1400

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
nErr = []
for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["ArtistName"]
    artistID = row["MusicBrainz"]
    mbID = row["MBRef"]
    if searchedForMasterArtists.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoResults(artistName=artistName, mbID=mbID)
    if response is None:
        print("Error ==> {0}".format((artistID,mbID,artistName)))
        searchedForErrors[artistID] = True
        apiio.sleep(15)
        nErr.append(artistID)
        if len(nErr) >= 6:
            break
        continue

    nErr = []
    searchedForMasterArtistsData[artistID] = response
    searchedForMasterArtists[artistID] = artistName
    apiio.sleep(20)
    n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        errors.save(data=searchedForErrors)
        print("="*150)
        apiio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
            
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)
if len(nErr) > 0:
    errors.save(data=searchedForErrors)
    for artistID in nErr:
        print("del searchedForErrors['{0}']".format(artistID))
    print("errors.save(data=searchedForErrors)")

Process [Getting SetListFM ArtistIDs] Start    ==> Time Is 2022-04-20 11:29:24
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-04-21 11:00:00 <====
   ====> Will Terminate Process 23 Hours and 30 Minutes From Now
Searching For The Lures (3e466c42-6c6e-483b-b35b-c81b9ccd72dd)                                True
Searching For 50 Shots Beats (6efc92d2-9cf9-4e79-8a03-47030fbc2d8b)                           

HTTP exception: HTTPError('404 Client Error: Not Found for url: https://api.setlist.fm/rest/1.0/artist/6efc92d2-9cf9-4e79-8a03-47030fbc2d8b')


==> Error in SetListFM search for 50 Shots Beats
Error ==> ('227676467019714471467562361765881577163', '6efc92d2-9cf9-4e79-8a03-47030fbc2d8b', '50 Shots Beats')
Searching For PRIZM (0d0230d3-c171-47c7-83bf-2c118ee4c2bf)                                    True
Searching For Star Pilots (7ca6dde6-936d-4eec-b027-188c219750c6)                              True
Searching For Emil Assergård (c6b74847-fb24-4f7f-b713-7e5146367857)                          True
Searching For Adramelech (259e6968-a841-4fc5-b02b-54665b85977b)                               True
5/?        : Process [Getting SetListFM ArtistIDs] Has Run For 2 Minutes and 1 Second.
Saving 23509 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For The Night Game (7266c2d8-3329-488b-8fa2-24fd9576939d)                           True
Searching For Elettra Lamborghini (f3a31b90-606a-4305-9e0f-6e7407efc8b0)                      True
Searching For Jenn Champion (47553783-12bd-4f2c-999c-d5743be93668)                            True
Searching For Bestia Bebé (b91a3d2a-bd11-46fe-938d-e4c57bc1df4b)                             True
Searching For Finger 5 (554bbe6f-6239-477c-bf5b-645e0eb77310)                                 True
10/?       : Process [Getting SetListFM ArtistIDs] Has Run For 3 Minutes and 55 Seconds.
Saving 23514 SetListFM Searched For Artist IDs


...

Searching For Elderwind (c5259e3a-b661-4687-b7ff-4383bd2c6e1b)                                True
Searching For Coconut Records (eddc0911-21fc-4327-ab90-ccc459ce1ef7)                          True
Searching For Mandy Capristo (e610fdbc-ce55-49f1-8484-6cfa4420bcdd)                           True
Searching For Peccatum (1a410219-6d65-4aba-87dc-b0fca2f50658)                                 True
Searching For Los Totora (20b9c72e-7282-4811-a39e-47c0ec840b58)                               True
15/?       : Process [Getting SetListFM ArtistIDs] Has Run For 5 Minutes and 49 Seconds.
Saving 23519 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For Monarch! (fce73a93-bd0b-4449-b275-eade0625f2c4)                                 True
Searching For nthng (83782be4-ceca-4eb9-8e40-4bf9c3e03471)                                    True
Searching For Loathe (56eb02c4-1f16-4613-8bb3-b4a752283fc3)                                   True
Searching For Patricia Lalor (4ed6d3e1-c5c5-4b33-988d-01a6af9b9ada)                           True
Searching For Stephen Jerzak (9f4453b8-46d3-4e49-9046-7a320deb587f)                           True
20/?       : Process [Getting SetListFM ArtistIDs] Has Run For 7 Minutes and 42 Seconds.
Saving 23524 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For Waking The Cadaver (80126582-98d9-4b0c-b846-6a67c1eb4b13)                       True
Searching For Mesarthim (910506c5-f932-47ce-a617-d9812614be2f)                                True
Searching For Polaris (b695cfa1-9804-4d33-9b53-7744c9e0c73a)                                  True
Searching For USA For Africa (45e3c90f-2de6-4452-99b3-8dbf7a3cf6df)                           True
Searching For Miss Pooja (e513b3f9-87a0-4b7a-9321-8d7fbc81e187)                               True


In [ ]:

del searchedForErrors['266123216221596336667210841006957488329']
del searchedForErrors['80109685308713840715995872602040319631']
del searchedForErrors['236385072074426569167334460189603196589']
del searchedForErrors['33974575013364044820404278826855531832']
del searchedForErrors['114542190165529234088310131782548142436']
del searchedForErrors['24451377252051597484574255839036437654']
errors.save(data=searchedForErrors)

## Save Results

In [6]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [7]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF.drop_duplicates(keep='first')
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

Found 1400 New Artists
Found 22104 Previous Artists
Found 23504 Total Results
Found 23504 Unique Results
Saving Data


In [8]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
mio.data.getRawArtistInfoData(mio.mv.get(3540473060), 3540473060)

In [ ]:
localArtistsDict

In [ ]:
localArtists.save(data=localArtistsDict)


In [ ]:
tracks
disc_count